In [ ]:

import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from sdlarch_rl.utils.utils import get_last_index, RealExcludeButtonsWrapper, GenericCNN, TimeLimit, FrameSkip
from sdlarch_rl import make
from stable_baselines3.common.atari_wrappers import WarpFrame
import time
import cv2
import numpy as np
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from pathlib import Path
import pygame
from inputs import get_gamepad, devices
import threading
from imitation.data.types import Trajectory
import torch as th
import os
import re
from utils import get_last_index, LSTMWrapper
import gymnasium as gym
from final_fight import FinalFightActionWrapper, make_env
import final_fight
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecTransposeImage
import math

env = make_env(human=False)()

# --- CONFIGS ---
MAX_TRAJ = 0
demo_path = 'demos/'
SCALE = 1/4
SCREEN_WIDTH = 854 # int(640 * SCALE)
SCREEN_HEIGHT = 480 # int(480 * SCALE)
NUMBER_OF_ACTIONS = 6
MAX_FPS=30
os.makedirs(demo_path, exist_ok=True)

# bc config
train_path = "./imitation/"
sub_train_path = train_path + "steps"
last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")
current_epoch = last_index_imitation

MAX_TRAJ = last_index_imitation

train_path="./imitation/"
sub_train_path = train_path + "steps"

#TODO comment
current_epoch = 20


last_index = -1
for p in Path(demo_path).iterdir():
    m = re.search(r"demos(\d+)\.pt$", p.name)
    if m: last_index = max(last_index, int(m.group(1)))


def make_policy(current_epoch):
    latest_model_path = sub_train_path + f"/bc_policy{current_epoch}.zip"
    policy = ActorCriticPolicy.load(latest_model_path)
    
    policy = LSTMWrapper(policy)
    policy.reset()


    return policy

count_record  = 0

better_epoch=-math.inf

while count_record < MAX_TRAJ:
    my_policy = make_policy(current_epoch)
    

    total_reward = 0

    print(f"Eval epoch: {current_epoch}")
    
    for episode in range(10):
        obs, _ = env.reset()
        my_policy.model.features_extractor.reset_hidden()
        done = False
        
        
        while not done:
            pred_act, _ = my_policy.predict(obs, deterministic=False)
            obs, reward, terminated, truncated, _ = env.step(pred_act)
            done = terminated or truncated
            total_reward += reward

        print(f"Total Acc Reward: {total_reward}")
    total_reward = total_reward/10
    print(f"Total Reward: {total_reward}")
    

    if total_reward > better_epoch:
        better_epoch = total_reward
        print(f"Better Epoch: {current_epoch}\n")

    current_epoch += 1
        
    if current_epoch > last_index_imitation:
        break
    


D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)


Eval epoch: 20
Total Acc Reward: 24.131999999999678
Total Acc Reward: 14.130000000005223
Total Acc Reward: 6.978000000005642
Total Acc Reward: 17.052000000009134
Total Acc Reward: 38.20600000000794
Total Acc Reward: 50.22299999999941
Total Acc Reward: 85.55800000000461
Total Acc Reward: 115.72400000001821
Total Acc Reward: 135.6740000000154
Total Acc Reward: 152.05699999999814
Total Reward: 15.205699999999814
Better Epoch: 20

Eval epoch: 21
Total Acc Reward: 30.998000000002314
Total Acc Reward: 44.99600000000205
Total Acc Reward: 33.99399999998616
Total Acc Reward: 19.588999999990946
Total Acc Reward: 64.58699999999622
Total Acc Reward: 66.58500000002653
Total Acc Reward: 77.58300000002353
Total Acc Reward: 79.86600000003219
Total Acc Reward: 82.39400000004494
Total Acc Reward: 76.39200000007524
Total Reward: 7.639200000007524
Eval epoch: 22
Total Acc Reward: 7.344999999999548
Total Acc Reward: 3.9799999999996447
Total Acc Reward: -6.701999999999947
Total Acc Reward: -17.4130000000001